# A11. 베이지안 업데이트 (Bayesian Update)

**데이터를 관찰할 때마다 나의 믿음(확률)이 어떻게 변하는지 실시간으로 관찰한다.**

> *"사실이 바뀌면, 나는 생각을 바꿉니다. 당신은 어떻게 하십니까?"*  
> — **존 메이너드 케인즈 (John Maynard Keynes)**

$$
P(\theta \mid \text{data}) = \frac{P(\text{data} \mid \theta) \cdot P(\theta)}{P(\text{data})}
$$

| 용어 | 기호 | 의미 |
|:---|:---:|:---|
| 사전확률 (Prior) | $P(\theta)$ | 데이터를 보기 전의 믿음 |
| 우도 (Likelihood) | $P(\text{data} \mid \theta)$ | 주어진 $\theta$ 하에서 데이터가 나올 확률 |
| 사후확률 (Posterior) | $P(\theta \mid \text{data})$ | 데이터를 본 후 업데이트된 믿음 |
| 증거 (Evidence) | $P(\text{data})$ | 정규화 상수 |

In [ ]:
# === 공통 설정 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 재현성
np.random.seed(42)

print("라이브러리 로드 완료")

---
## 베이지안 사고 도입

### 빈도주의 vs 베이지안

| | 빈도주의 (Frequentist) | 베이지안 (Bayesian) |
|:---|:---|:---|
| **확률의 의미** | 장기적 빈도 | 주관적 믿음의 정도 |
| **모수** | 고정된 미지의 상수 | 불확실성을 가진 확률변수 |
| **데이터** | 확률변수 | 관찰된 고정값 |
| **추론 방법** | MLE, 신뢰구간 | 사전분포 → 사후분포 |

### 동전 던지기 예시

새 동전을 받았다. 이 동전이 공정한지(앞면 확률 = 0.5) 알고 싶다.

- **처음**: 아무 정보 없음 → 확률 $p$가 0~1 사이 어디든 가능 (균등 Prior)
- **10번 던져서 7번 앞면**: 약간 앞면에 치우친 동전일까?
- **100번 던져서 70번 앞면**: 거의 확실히 $p \approx 0.7$

**데이터가 쌓일수록 불확실성이 줄어들고, 믿음이 수렴한다.**

---
## Beta 분포를 사전분포로 사용

동전의 앞면 확률 $p$에 대한 믿음을 **Beta 분포**로 표현한다.

$$
p \sim \text{Beta}(\alpha, \beta)
$$

| 상태 | $\alpha$ | $\beta$ | 의미 |
|:---|:---:|:---:|:---|
| 아무것도 모름 | 1 | 1 | 균등분포 (모든 $p$ 동일 가능성) |
| 앞면 5번 관찰 | 1+5=6 | 1 | 앞면 쪽으로 편향 |
| 앞면 5번, 뒷면 3번 | 6 | 4 | 약간 앞면 우세 |
| 앞면 50번, 뒷면 50번 | 51 | 51 | $p \approx 0.5$에 집중 |

### 업데이트 규칙 (Conjugate Prior)

$$
\text{Prior: } \text{Beta}(\alpha, \beta) \xrightarrow{\text{앞면 } h \text{번, 뒷면 } t \text{번}} \text{Posterior: } \text{Beta}(\alpha + h, \beta + t)
$$

**Beta는 이항 분포의 켤레 사전분포(Conjugate Prior)**이므로, 사후분포도 Beta가 된다!

In [ ]:
# Beta 분포의 다양한 형태 시각화
x = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) 다양한 Prior
ax = axes[0]
priors = [(1, 1, '아무것도 모름'), (2, 2, '약간 0.5 선호'),
          (5, 5, '0.5에 확신'), (2, 5, '뒷면 편향 예상')]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
for (a, b, label), c in zip(priors, colors):
    ax.plot(x, stats.beta.pdf(x, a, b), linewidth=2.5, color=c,
            label=f'Beta({a},{b}): {label}')
ax.set_xlabel('p (앞면 확률)', fontsize=12)
ax.set_ylabel('확률밀도', fontsize=12)
ax.set_title('다양한 사전분포 (Prior)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)

# (2) 관찰에 따른 업데이트
ax = axes[1]
observations = [(0, 0), (3, 1), (7, 3), (35, 15), (70, 30)]
colors2 = ['#bdc3c7', '#3498db', '#2ecc71', '#e67e22', '#e74c3c']
for (h, t), c in zip(observations, colors2):
    a, b = 1 + h, 1 + t
    ax.plot(x, stats.beta.pdf(x, a, b), linewidth=2.5, color=c,
            label=f'앞{h}뒤{t} → Beta({a},{b})')
ax.axvline(0.7, color='red', linestyle=':', linewidth=1.5, alpha=0.5, label='실제 p=0.7')
ax.set_xlabel('p (앞면 확률)', fontsize=12)
ax.set_ylabel('확률밀도', fontsize=12)
ax.set_title('데이터가 쌓이면 → 진실로 수렴', fontsize=14, fontweight='bold')
ax.legend(fontsize=8)

# (3) 불확실성 감소
ax = axes[2]
n_obs = [0, 1, 5, 10, 50, 100, 500]
std_devs = []
for n in n_obs:
    h = int(n * 0.7)  # 실제 p=0.7
    t = n - h
    a, b = 1 + h, 1 + t
    std_devs.append(stats.beta.std(a, b))
ax.plot(n_obs, std_devs, 'o-', color='#e74c3c', linewidth=2.5, markersize=10)
for i, (n, s) in enumerate(zip(n_obs, std_devs)):
    ax.annotate(f'{s:.3f}', (n, s), textcoords="offset points",
                xytext=(10, 10), fontsize=9)
ax.set_xlabel('관찰 횟수', fontsize=12)
ax.set_ylabel('사후분포 표준편차', fontsize=12)
ax.set_title('관찰이 많을수록 불확실성 감소', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 해석

- **Prior (사전분포)**: 관찰 전 믿음 — Beta(1,1)은 "아무것도 모른다"는 뜻
- **데이터가 쌓이면**: 분포가 점점 좁아지고, 실제 확률(p=0.7) 근처로 수렴
- **불확실성 감소**: 관찰 100회 후에는 표준편차가 0.04 수준으로 매우 작아진다

---
## Matplotlib 버전: 단계별 업데이트 시각화

실제 동전을 던지는 시뮬레이션을 수행하고, 각 단계에서 사후분포가 어떻게 변하는지 관찰한다.

In [ ]:
# 시뮬레이션 설정
true_p = 0.65  # 실제 동전의 앞면 확률 (알 수 없다고 가정)
np.random.seed(42)

# 단계별 관찰 횟수
checkpoints = [0, 1, 5, 10, 50, 100, 500]

# 동전 던지기 데이터 생성
max_n = max(checkpoints)
flips = np.random.binomial(1, true_p, max_n)  # 1=앞면, 0=뒷면

print(f"실제 앞면 확률: p = {true_p}")
print(f"총 {max_n}회 던지기 중 앞면: {flips.sum()}회 ({flips.sum()/max_n*100:.1f}%)")

In [ ]:
x = np.linspace(0, 1, 1000)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# 색상 그라데이션
cmap = plt.cm.RdYlBu_r
colors = [cmap(i / (len(checkpoints) - 1)) for i in range(len(checkpoints))]

for idx, n in enumerate(checkpoints):
    row = idx // 4
    col = idx % 4
    ax = axes[row][col]
    
    if n == 0:
        alpha_post, beta_post = 1, 1
        heads, tails = 0, 0
    else:
        heads = int(flips[:n].sum())
        tails = n - heads
        alpha_post = 1 + heads
        beta_post = 1 + tails
    
    # Posterior 그리기
    pdf = stats.beta.pdf(x, alpha_post, beta_post)
    ax.fill_between(x, pdf, color=colors[idx], alpha=0.3)
    ax.plot(x, pdf, color=colors[idx], linewidth=2.5)
    
    # 실제 p 표시
    ax.axvline(true_p, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    # MAP (최빈값) 표시
    if alpha_post > 1 and beta_post > 1:
        mode = (alpha_post - 1) / (alpha_post + beta_post - 2)
        ax.axvline(mode, color='blue', linestyle=':', linewidth=1.5, alpha=0.7)
    
    # 95% HDI (Highest Density Interval)
    hdi_low, hdi_high = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
    ax.axvspan(hdi_low, hdi_high, color='gray', alpha=0.1)
    
    if n == 0:
        title = f'n=0 (Prior)\nBeta(1,1)'
    else:
        title = f'n={n} (앞{heads}, 뒤{tails})\nBeta({alpha_post},{beta_post})'
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('p', fontsize=10)
    ax.set_ylabel('확률밀도', fontsize=10)
    ax.set_xlim(0, 1)
    
    # 95% HDI 범위 텍스트
    ax.text(0.95, 0.95, f'95% HDI\n[{hdi_low:.3f}, {hdi_high:.3f}]',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# 마지막 빈 칸에 범례
ax_legend = axes[1][3]
ax_legend.axis('off')
ax_legend.text(0.5, 0.7, '범례', fontsize=14, fontweight='bold',
               ha='center', transform=ax_legend.transAxes)
ax_legend.plot([], [], 'r--', linewidth=2, label=f'실제 p = {true_p}')
ax_legend.plot([], [], 'b:', linewidth=1.5, label='MAP (최빈값)')
ax_legend.fill_between([], [], color='gray', alpha=0.3, label='95% HDI')
ax_legend.legend(fontsize=12, loc='center')
ax_legend.text(0.5, 0.15,
               '관찰이 많아질수록\n분포가 좁아지고\n실제 p에 수렴!',
               fontsize=11, ha='center', va='bottom',
               transform=ax_legend.transAxes,
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle('베이지안 업데이트: Prior → Posterior 진화 과정',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **n=0 (Prior)**: Beta(1,1) = 균등분포 — "아무것도 모르는" 상태
- **n=1~5**: 분포가 서서히 형태를 갖추기 시작하지만 아직 넓다 (불확실성 높음)
- **n=10~50**: 분포가 점점 좁아지며 실제 p 근처로 이동
- **n=100~500**: 분포가 매우 좁아져서 실제 p를 거의 정확하게 추정
- **95% HDI**: 관찰이 쌓일수록 구간이 좁아진다 → 확신도 증가

> **핵심**: 베이지안 추론은 "정답을 맞추는 것"이 아니라,  
> **데이터에 기반하여 불확실성을 줄여가는 과정**이다.

---
## 실시간 업데이트 추적

In [ ]:
# 매 던지기마다의 추정 확률 변화 추적
np.random.seed(42)
n_total = 200
true_p_track = 0.65
flips_track = np.random.binomial(1, true_p_track, n_total)

# 매 시점의 MAP 추정, 95% HDI
maps = []
hdi_lows = []
hdi_highs = []
freq_estimates = []

for i in range(1, n_total + 1):
    h = int(flips_track[:i].sum())
    t = i - h
    a = 1 + h
    b = 1 + t
    
    # MAP (최빈값)
    if a > 1 and b > 1:
        mode = (a - 1) / (a + b - 2)
    else:
        mode = h / i if i > 0 else 0.5
    maps.append(mode)
    
    # 95% HDI
    low, high = stats.beta.ppf([0.025, 0.975], a, b)
    hdi_lows.append(low)
    hdi_highs.append(high)
    
    # 빈도주의 추정
    freq_estimates.append(h / i)

n_range = np.arange(1, n_total + 1)

fig, ax = plt.subplots(figsize=(16, 6))

# 95% HDI 밴드
ax.fill_between(n_range, hdi_lows, hdi_highs, color='#3498db', alpha=0.15, label='95% HDI')

# MAP 추정 곡선
ax.plot(n_range, maps, color='#3498db', linewidth=2, label='베이지안 MAP 추정')

# 빈도주의 추정
ax.plot(n_range, freq_estimates, color='#e67e22', linewidth=1.5, alpha=0.6,
        linestyle='--', label='빈도주의 추정 (h/n)')

# 실제 p
ax.axhline(true_p_track, color='red', linestyle='-', linewidth=2.5,
           label=f'실제 p = {true_p_track}', alpha=0.7)

ax.set_xlabel('관찰 횟수 (n)', fontsize=13)
ax.set_ylabel('p 추정값', fontsize=13)
ax.set_title('베이지안 업데이트: 실시간 추정 수렴 과정', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(1, n_total)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"n=10 시점: MAP={maps[9]:.3f}, 95% HDI=[{hdi_lows[9]:.3f}, {hdi_highs[9]:.3f}]")
print(f"n=50 시점: MAP={maps[49]:.3f}, 95% HDI=[{hdi_lows[49]:.3f}, {hdi_highs[49]:.3f}]")
print(f"n=200 시점: MAP={maps[-1]:.3f}, 95% HDI=[{hdi_lows[-1]:.3f}, {hdi_highs[-1]:.3f}]")

---
## Gradio UI 버전: 인터랙티브 베이지안 업데이트

슬라이더로 동전의 실제 확률과 관찰 횟수를 조절하며 베이지안 업데이트를 체험한다.

In [ ]:
try:
    import gradio as gr
    HAS_GRADIO = True
    print("gradio 버전:", gr.__version__)
except ImportError:
    HAS_GRADIO = False
    print("gradio가 설치되어 있지 않습니다.")
    print("설치 명령: pip install gradio")
    print("\n→ Gradio 없이 matplotlib 정적 버전으로 대체합니다.")

In [ ]:
if HAS_GRADIO:
    def bayesian_experiment(true_prob, n_observations, seed):
        """
        베이지안 동전 실험
        
        Parameters:
            true_prob: 실제 동전의 앞면 확률
            n_observations: 관찰 횟수
            seed: 랜덤 시드
        """
        np.random.seed(int(seed))
        n_observations = int(n_observations)
        
        # 동전 던지기
        if n_observations > 0:
            flips = np.random.binomial(1, true_prob, n_observations)
            heads = int(flips.sum())
            tails = n_observations - heads
        else:
            heads, tails = 0, 0
        
        # Beta 파라미터
        alpha_post = 1 + heads
        beta_post = 1 + tails
        
        # 그래프 생성
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        x = np.linspace(0, 1, 500)
        
        # (1) Prior vs Posterior
        ax = axes[0]
        ax.plot(x, stats.beta.pdf(x, 1, 1), 'k--', linewidth=1.5,
                alpha=0.5, label='Prior: Beta(1,1)')
        pdf = stats.beta.pdf(x, alpha_post, beta_post)
        ax.fill_between(x, pdf, color='#3498db', alpha=0.3)
        ax.plot(x, pdf, color='#3498db', linewidth=2.5,
                label=f'Posterior: Beta({alpha_post},{beta_post})')
        ax.axvline(true_prob, color='red', linestyle='-', linewidth=2,
                   label=f'실제 p = {true_prob}')
        
        # MAP
        if alpha_post > 1 and beta_post > 1:
            mode = (alpha_post - 1) / (alpha_post + beta_post - 2)
            ax.axvline(mode, color='green', linestyle=':', linewidth=1.5,
                       label=f'MAP = {mode:.3f}')
        
        ax.set_xlabel('p (앞면 확률)', fontsize=12)
        ax.set_ylabel('확률밀도', fontsize=12)
        ax.set_title('사후분포 (Posterior)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xlim(0, 1)
        
        # (2) 누적 추정 과정
        ax = axes[1]
        if n_observations > 0:
            cum_heads = np.cumsum(flips)
            cum_n = np.arange(1, n_observations + 1)
            cum_map = []
            cum_hdi_low = []
            cum_hdi_high = []
            for i in range(n_observations):
                a = 1 + int(cum_heads[i])
                b = 1 + (i + 1) - int(cum_heads[i])
                if a > 1 and b > 1:
                    cum_map.append((a - 1) / (a + b - 2))
                else:
                    cum_map.append(int(cum_heads[i]) / (i + 1))
                low, high = stats.beta.ppf([0.025, 0.975], a, b)
                cum_hdi_low.append(low)
                cum_hdi_high.append(high)
            
            ax.fill_between(cum_n, cum_hdi_low, cum_hdi_high,
                            color='#3498db', alpha=0.15)
            ax.plot(cum_n, cum_map, color='#3498db', linewidth=2)
        
        ax.axhline(true_prob, color='red', linewidth=2, alpha=0.7)
        ax.set_xlabel('관찰 횟수', fontsize=12)
        ax.set_ylabel('p 추정', fontsize=12)
        ax.set_title('수렴 과정', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1)
        
        plt.tight_layout()
        
        # 결과 텍스트
        mean_post = alpha_post / (alpha_post + beta_post)
        std_post = stats.beta.std(alpha_post, beta_post)
        hdi_low, hdi_high = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
        
        result = f"""실험 결과
{'='*40}
관찰: {n_observations}회 (앞면 {heads}회, 뒷면 {tails}회)
사후분포: Beta({alpha_post}, {beta_post})
추정 평균: {mean_post:.4f}
추정 표준편차: {std_post:.4f}
95% HDI: [{hdi_low:.4f}, {hdi_high:.4f}]
실제 p: {true_prob}
오차: {abs(mean_post - true_prob):.4f}"""
        
        return fig, result
    
    # Gradio UI
    demo = gr.Interface(
        fn=bayesian_experiment,
        inputs=[
            gr.Slider(0.01, 0.99, value=0.65, step=0.01,
                      label="실제 동전 확률 p"),
            gr.Slider(0, 1000, value=50, step=1,
                      label="관찰 횟수 n"),
            gr.Number(value=42, label="랜덤 시드"),
        ],
        outputs=[
            gr.Plot(label="베이지안 업데이트 시각화"),
            gr.Textbox(label="실험 결과", lines=10),
        ],
        title="베이지안 동전 실험",
        description="슬라이더를 조절하여 관찰 횟수에 따른 베이지안 업데이트를 체험하세요.",
        examples=[
            [0.5, 10, 42],
            [0.7, 100, 42],
            [0.3, 500, 42],
        ],
    )
    demo.launch(inline=True)
else:
    print("Gradio 미설치 — 위의 matplotlib 버전 결과를 참고하세요.")
    print("\n대체: 3가지 시나리오를 정적으로 보여드립니다.")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    x = np.linspace(0, 1, 500)
    
    scenarios = [
        (0.5, 10, '공정한 동전, 10회'),
        (0.7, 100, '편향 동전, 100회'),
        (0.3, 500, '약한 편향, 500회'),
    ]
    
    for idx, (p, n, title) in enumerate(scenarios):
        ax = axes[idx]
        np.random.seed(42)
        flips_s = np.random.binomial(1, p, n)
        h = int(flips_s.sum())
        t = n - h
        a, b = 1 + h, 1 + t
        
        ax.plot(x, stats.beta.pdf(x, 1, 1), 'k--', alpha=0.4, label='Prior')
        pdf = stats.beta.pdf(x, a, b)
        ax.fill_between(x, pdf, alpha=0.3)
        ax.plot(x, pdf, linewidth=2.5, label=f'Beta({a},{b})')
        ax.axvline(p, color='red', linestyle='--', linewidth=2, label=f'실제 p={p}')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('p', fontsize=11)
        ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()

---
## 한강에서 가장 큰 물고기의 크기는 몇 cm?

### 문제 설정

당신은 한강에서 낚시를 한다. 잡은 물고기의 크기를 기록하며,  
"한강에서 가장 큰 물고기는 몇 cm일까?"를 베이지안으로 추정해 보자.

- **사전지식**: 한강 물고기는 보통 10~80cm 정도? (매우 불확실)
- **관찰 데이터**: 낚시에서 잡힌 물고기 크기
- **목표**: 최대 크기 $M$의 사후분포를 추정

### 베이지안 접근

$$
M \sim \text{Prior: Normal}(\mu_0, \sigma_0^2)
$$

관찰 데이터 $x_1, x_2, \ldots, x_n$을 보면서 $M$에 대한 믿음을 업데이트한다.

In [ ]:
# 한강 물고기 시뮬레이션
np.random.seed(42)

# 실제 물고기 크기 분포 (우리는 모른다고 가정)
# 대부분 작고, 아주 가끔 큰 물고기 → 로그정규 분포
true_fish_sizes = np.random.lognormal(mean=3.0, sigma=0.6, size=10000)
# cm 단위 (대부분 10~50cm, 가끔 80cm+)

print(f"한강 물고기 모집단 (시뮬레이션)")
print(f"총 물고기 수: {len(true_fish_sizes):,}")
print(f"평균 크기: {true_fish_sizes.mean():.1f} cm")
print(f"중앙값 크기: {np.median(true_fish_sizes):.1f} cm")
print(f"최대 크기: {true_fish_sizes.max():.1f} cm")
print(f"상위 1% 크기: {np.percentile(true_fish_sizes, 99):.1f} cm")

In [ ]:
# 낚시 시뮬레이션: 순차적으로 물고기를 잡으며 최대 크기 추정
np.random.seed(42)

# 잡힌 물고기 순서
caught_fish = np.random.choice(true_fish_sizes, size=50, replace=False)

# 사전분포: 최대 크기에 대한 Prior
# "한강 물고기는 아마 최대 50cm 정도?" → Normal(50, 30^2)
prior_mean = 50
prior_std = 30

# 관찰하면서 믿음 업데이트
checkpoints_fish = [1, 3, 5, 10, 20, 50]
x_range_fish = np.linspace(0, 200, 500)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, n in enumerate(checkpoints_fish):
    ax = axes[idx]
    observed = caught_fish[:n]
    max_observed = observed.max()
    
    # 간단한 베이지안 업데이트: 관찰된 최대값 기반
    # Posterior mean = (prior_precision * prior_mean + data_precision * data_mean) / total_precision
    # 여기서는 관찰된 최대값을 데이터의 "증거"로 사용
    n_weight = n * 0.5  # 관찰 가중치 (조절 가능)
    post_precision = 1/prior_std**2 + n_weight/prior_std**2
    post_mean = (prior_mean/prior_std**2 + max_observed * n_weight/prior_std**2) / post_precision
    post_std = np.sqrt(1 / post_precision)
    
    # Prior
    ax.plot(x_range_fish, stats.norm.pdf(x_range_fish, prior_mean, prior_std),
            'k--', linewidth=1, alpha=0.4, label='Prior')
    
    # Posterior
    pdf = stats.norm.pdf(x_range_fish, post_mean, post_std)
    ax.fill_between(x_range_fish, pdf, color='#2ecc71', alpha=0.3)
    ax.plot(x_range_fish, pdf, color='#2ecc71', linewidth=2.5,
            label=f'Posterior (평균={post_mean:.1f})')
    
    # 관찰된 최대값
    ax.axvline(max_observed, color='blue', linestyle=':', linewidth=2,
               label=f'관찰 최대: {max_observed:.1f}cm')
    
    # 실제 최대값
    ax.axvline(true_fish_sizes.max(), color='red', linestyle='--', linewidth=1.5,
               alpha=0.5, label=f'실제 최대: {true_fish_sizes.max():.1f}cm')
    
    # 잡힌 물고기 표시
    ax.scatter(observed, np.zeros(len(observed)) - 0.001,
               marker='|', s=100, color='orange', zorder=5, linewidth=2)
    
    ax.set_title(f'{n}마리 관찰', fontsize=13, fontweight='bold')
    ax.set_xlabel('물고기 크기 (cm)', fontsize=10)
    ax.set_ylabel('확률밀도', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(0, 180)

fig.suptitle('한강 최대 물고기 크기: 베이지안 순차 추정',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **1마리만 잡았을 때**: Prior의 영향이 크다 — 사전 믿음에 많이 의존
- **10마리 잡았을 때**: 관찰 데이터가 Prior를 압도하기 시작
- **50마리 잡았을 때**: 관찰된 최대값 근처로 확신이 높아진다

### "정답은 없다, 데이터에 따라 믿음이 진화한다"

1. **사전지식이 틀려도 괜찮다**: 데이터가 충분하면 Prior의 영향은 사라진다
2. **적은 데이터에서는 Prior가 중요**: 관찰이 적을 때 합리적 사전지식은 도움이 된다
3. **완벽한 추정은 불가능**: 우리가 잡지 못한 더 큰 물고기가 있을 수 있다
4. **불확실성을 인정하는 것이 과학적 태도**: 점추정(하나의 숫자)보다 분포(불확실성 포함)가 정직하다

> *"데이터가 바뀌면, 믿음도 바뀌어야 한다."*  
> 이것이 베이지안 사고의 핵심이다.

---
## 종합 정리

### 베이지안 업데이트의 3가지 핵심

| | 내용 |
|:---|:---|
| **Prior** | 데이터 전의 믿음 (주관적이어도 된다) |
| **Likelihood** | 데이터가 모델을 얼마나 지지하는가 |
| **Posterior** | 데이터 후 업데이트된 믿음 |

### 핵심 통찰

1. **데이터가 충분하면 Prior는 상관없다**: 서로 다른 사전 믿음도 같은 데이터를 보면 수렴
2. **적은 데이터에서 Prior는 정규화 역할**: 과적합(Overfitting) 방지
3. **불확실성을 정량화**: "p = 0.7"이 아니라 "p는 0.65~0.75 사이일 가능성이 95%"

### 실제 응용

| 분야 | 베이지안 응용 |
|:---|:---|
| 스팸 필터 | Naive Bayes 분류기 |
| A/B 테스트 | 베이지안 A/B 테스트 (early stopping 가능) |
| 딥러닝 | Bayesian Neural Networks, Dropout as Bayes |
| 추천 시스템 | Thompson Sampling (다중 슬롯머신 문제) |
| 의학 | 검사 양성 시 실제 질병 확률 (베이즈 정리) |